# Lab 3 — xAI con XGBoost (SHM) — Vía IA-asistida

**Sesión:** inteligencia artificial explicable aplicada a sensores estructurales

**Público:** ingenieros civiles **sin experiencia en programación** · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar (Copilot, Gemini, Cursor, etc.)

1. Abre este repo en **Codespaces** o activa `source labs/.venv/bin/activate`.
2. **Ejecuta** las celdas pre-escritas (carga de datos, gráficos base) en orden.
3. Lee la **guía IA** de la sección: objetivo, qué considerar y el prompt sugerido.
4. Copia el prompt a tu asistente; pega el código generado en la celda **«Aquí coloca tu solución»**.
5. Ejecuta tu celda y la **Autoevaluación**; busca ✅ antes de avanzar.
6. Registra tus prompts en [`prompts_entregados.md`](prompts_entregados.md) (entrega obligatoria).
7. Referencia docente: `xai_estructuras_solucion.ipynb` (no distribuir al inicio).

**La IA propone; tú validas** con autoevaluación, gráficos y sentido físico/estructural.

Dataset: [`data/DATOS.md`](data/DATOS.md) — mismas columnas que Lab 1.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_panorama_xai, verificar_carga, verificar_features, verificar_limpieza,
    verificar_columna, verificar_etiquetas, verificar_xgboost, verificar_importancia_global,
    verificar_shap_global, verificar_shap_local, verificar_lime_local, verificar_pdp,
    resumen_seccion,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (Codespaces / labs/.venv).')


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración en tres ejes |
| Strain | με | Deformación (extensómetro) |
| Temp | °C | Temperatura del sensor |
| **Condition Label** | **0 / 1 / 2** | **Normal / daño menor / severo** |

Mismo CSV que **Lab 1**. Entrenamos **XGBoost** y aplicamos un **kit xAI**: importancia, permutation, **SHAP**, **LIME** y **PDP**.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Panorama xAI en monitoreo estructural

El objetivo central del lab **no es solo entrenar** el clasificador: es **probar varias técnicas de explicación** sobre el mismo modelo.

### Caja de herramientas xAI (este lab)

| Técnica | Alcance | Sección | Idea clave |
|---------|---------|---------|------------|
| **Importancia del booster** | Global | 7 | Qué sensores usa el árbol en promedio |
| **Permutation importance** | Global | 7 (pre) | Cuánto cae el rendimiento si barajas una feature |
| **SHAP** (`TreeExplainer`) | Global + local | 8–9 | Contribución marginal por sensor (eficiente en árboles) |
| **LIME** | Local | 10 | Aproximación interpretable de **un caso** (comparar con SHAP) |
| **PDP + SHAP dependence** | Global marginal | 11 | Efecto promedio de un sensor vs interacciones |

### 📘 Subpreguntas
- **1.a** ¿Qué aporta xAI frente a solo mirar accuracy?
- **1.b** ¿Cuándo usarías SHAP vs LIME para explicar una alerta en obra?
- **1.c** ¿Qué diferencia hay entre explicación **global** y **local**?


#### ✍️ Tu respuesta (opcional, 2–3 líneas)




### 🤖 Guía IA — Sección 1: Panorama xAI

**Objetivo:** Listar las técnicas de explicación que aplicarás en este lab (kit completo).

**Tu código debe lograr**
- Definir `TECNICAS_XAI` como lista con al menos 4 técnicas
- Incluir importancia, shap, lime y pdp
- Imprimir la lista

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `TECNICAS_XAI`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Este lab aplica varias técnicas sobre el mismo XGBoost — no solo una
- Nombres sugeridos: importancia, permutation, shap, lime, pdp
- Global = importancia/permutation/PDP; local = SHAP waterfall y LIME

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Estoy en el Lab 3 de xAI con XGBoost y sensores SHM.
Genera código que:
1) defina TECNICAS_XAI = ["importancia", "permutation", "shap", "lime", "pdp"]
2) imprima "Técnicas xAI que aplicarás en este lab:"
3) recorra la lista e imprima cada técnica con print(f"  · {t}")
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · TECNICAS_XAI

# Tu código debe:
#   1. Lista con importancia, shap, lime, pdp (y opcional permutation)
#   2. Imprimir cada técnica



In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `TECNICAS_XAI`
r = []
try:
    r = verificar_panorama_xai(TECNICAS_XAI)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Panorama xAI', r)


## Pregunta 2 — Carga del dataset de sensores

### 📘 Subpreguntas
- **2.a** ¿Cuántas features de sensor vs 1 etiqueta de condición?
- **2.b** ¿Por qué excluimos `Timestamp` del modelo?


In [ ]:
# --- PRE-ESCRITO: carga ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")


### 🤖 Guía IA — Sección 2: Carga del dataset

**Objetivo:** Definir las 5 features de sensor y mostrar las primeras filas del CSV.

**Tu código debe lograr**
- Lista `FEATURES` con los 5 nombres exactos de columnas
- `N_FILAS_HEAD` entre 1 y 20
- Mostrar `df.head(N_FILAS_HEAD)` con display

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `FEATURES`, `N_FILAS_HEAD`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- La celda anterior ya cargó `df` desde `data/building_health_monitoring_dataset.csv` (1000×7)
- No incluir `Timestamp` ni `Condition Label` en FEATURES
- Nombres exactos: Accel_X (m/s^2), Accel_Y (m/s^2), Accel_Z (m/s^2), Strain (με), Temp (°C)

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter ya tengo `df` cargado (1000 filas, 7 columnas) del CSV de monitoreo estructural.
Genera código que:
1) defina FEATURES como lista con las 5 columnas de sensores (sin Timestamp ni Condition Label)
2) defina N_FILAS_HEAD = 5
3) imprima FEATURES
4) muestre df.head(N_FILAS_HEAD) con display()
Usa los nombres de columna exactos del dataset.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · FEATURES
#   · N_FILAS_HEAD

# Tu código debe:
#   1. Definir FEATURES (5 sensores, nombres exactos)
#   2. Definir N_FILAS_HEAD
#   3. Imprimir FEATURES y mostrar df.head()



In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `FEATURES`, `N_FILAS_HEAD`
r = []
try:
    r = verificar_carga(df, N_FILAS_HEAD)
    r.extend(verificar_features(FEATURES))
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Carga', r)


## Pregunta 3 — Calidad de datos y limpieza

### 📘 Subpreguntas
- **3.a** ¿Cuántas filas se pierden al eliminar nulos?
- **3.b** ¿Por qué `Strain` suele ser crítico en SHM?


In [ ]:
# --- PRE-ESCRITO: limpieza ---
n_antes = len(df)
n_nulos_por_col = df[FEATURES].isna().sum()
print("Nulos por sensor (crudo):")
display(n_nulos_por_col)
df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
print(f"Tras dropna: {n_antes} → {n_despues} lecturas")


### 🤖 Guía IA — Sección 3: Limpieza y revisión de sensor

**Objetivo:** Elegir un sensor y mostrar sus estadísticas en datos crudos.

**Tu código debe lograr**
- Definir `COLUMNA_REVISAR` con un nombre de sensor válido
- Calcular `.describe()` sobre `df[COLUMNA_REVISAR]`
- Mostrar el resultado

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `COLUMNA_REVISAR`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- La celda anterior ya hizo dropna y creó `df_limpio`, `n_antes`, `n_despues`
- Tu tarea es revisar un sensor en datos **crudos** (`df`), no en df_limpio
- Strain (με) es el sensor más relevante para daño estructural

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo `df` (crudo) y `df_limpio` (sin nulos en FEATURES).
Genera código que:
1) defina COLUMNA_REVISAR = "Strain (με)"
2) calcule stats_col = df[COLUMNA_REVISAR].describe()
3) imprima el nombre de la columna y muestre stats_col con display()
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · COLUMNA_REVISAR
#   · stats_col

# Tu código debe:
#   1. Elegir COLUMNA_REVISAR (sensor válido)
#   2. Calcular describe() en df
#   3. Imprimir y mostrar estadísticas



In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `COLUMNA_REVISAR`
r = []
try:
    r = verificar_limpieza(df_limpio, n_antes, n_despues)
    r.extend(verificar_columna(df, COLUMNA_REVISAR))
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Limpieza', r)


## Pregunta 4 — Balance de clases (Condition Label)

### 📘 Subpreguntas
- **4.a** ¿La clase 0 es mayoritaria? ¿Qué implica para el modelo?
- **4.b** ¿Por qué usamos `stratify` en el split?


In [ ]:
# --- PRE-ESCRITO: distribución de etiquetas ---
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(conteo).plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_xlabel('Condition Label')
ax.set_ylabel('Conteo')
ax.set_title('Distribución de estados estructurales')
plt.tight_layout()
plt.show()
print("Conteo:", conteo)


### 🤖 Guía IA — Sección 4: Balance de clases

**Objetivo:** Mostrar el conteo de las clases de Condition Label.

**Tu código debe lograr**
- Definir `N_CLASES_MOSTRAR` (1 a 3)
- Usar el dict `conteo` ya calculado
- Mostrar las primeras N clases ordenadas

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `N_CLASES_MOSTRAR`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- La celda anterior ya graficó y creó `conteo` desde df_limpio['Condition Label']
- Clases: 0=normal, 1=daño menor, 2=daño severo
- N_CLASES_MOSTRAR = 3 muestra las tres clases

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter ya existe conteo = dict con value_counts de Condition Label.
Genera código que:
1) defina N_CLASES_MOSTRAR = 3
2) cree serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
3) imprima cuántas clases muestra y display(serie_clases)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · N_CLASES_MOSTRAR
#   · serie_clases

# Tu código debe:
#   1. Definir N_CLASES_MOSTRAR
#   2. Construir serie desde conteo
#   3. Mostrar distribución



In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `N_CLASES_MOSTRAR`
r = []
try:
    r = verificar_etiquetas(conteo, N_CLASES_MOSTRAR)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Balance de clases', r)


## Pregunta 5 — Partición y XGBoost

### 📘 Subpreguntas
- **5.a** ¿Por qué escalamos sensores antes de XGBoost?
- **5.b** ¿Qué controlan `max_depth` y `learning_rate`?


In [ ]:
# --- PRE-ESCRITO: escalado y split estratificado ---
X_raw = df_limpio[FEATURES].values
y = df_limpio['Condition Label'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


### 🤖 Guía IA — Sección 5: Entrenar XGBoost

**Objetivo:** Configurar hiperparámetros y entrenar el clasificador multiclass.

**Tu código debe lograr**
- Definir N_ESTIMATORS, MAX_DEPTH, LEARNING_RATE en rangos válidos
- Crear XGBClassifier con objective='multi:softprob' y num_class=3
- Entrenar con modelo.fit(X_train, y_train)

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `modelo`, `N_ESTIMATORS`, `MAX_DEPTH`, `LEARNING_RATE`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- X_train, X_test, y_train, y_test y RANDOM_STATE=42 ya existen (split estratificado)
- Rangos válidos: n_estimators 10–500, max_depth 2–12, learning_rate 0.01–0.5
- Usar eval_metric='mlogloss' y random_state=RANDOM_STATE

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo X_train, X_test, y_train, y_test (datos escalados, stratify) y RANDOM_STATE=42.
Genera código que:
1) defina N_ESTIMATORS=100, MAX_DEPTH=6, LEARNING_RATE=0.1
2) cree modelo = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=..., max_depth=..., learning_rate=..., random_state=RANDOM_STATE, eval_metric='mlogloss')
3) entrene con modelo.fit(X_train, y_train)
4) imprima confirmación de entrenamiento
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · N_ESTIMATORS
#   · MAX_DEPTH
#   · LEARNING_RATE
#   · modelo

# Tu código debe:
#   1. Definir hiperparámetros
#   2. Instanciar XGBClassifier multiclass
#   3. Entrenar el modelo



In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `modelo`, `N_ESTIMATORS`, `MAX_DEPTH`, `LEARNING_RATE`
r = []
try:
    y_pred_tmp = modelo.predict(X_test)
    acc_tmp = accuracy_score(y_test, y_pred_tmp)
    r = verificar_xgboost(acc_tmp, N_ESTIMATORS, MAX_DEPTH, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — XGBoost', r)


## Pregunta 6 — Métricas de clasificación

### 📘 Subpreguntas
- **6.a** ¿Qué clases se confunden más en la matriz?
- **6.b** ¿El recall de daño severo (clase 2) es aceptable para alertas?


In [ ]:
# --- PRE-ESCRITO: predicciones en test ---
y_pred = modelo.predict(X_test)
acc_test = accuracy_score(y_test, y_pred)
print(f"Accuracy test: {acc_test:.3f}")


### 🤖 Guía IA — Sección 6: Métricas de clasificación

**Objetivo:** Matriz de confusión y classification_report.

**Tu código debe lograr**
- Construir matriz de confusión 3×3
- Graficar heatmap con seaborn
- Imprimir classification_report

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- modelo, X_test, y_test ya existen; la celda anterior calculó y_pred y acc_test
- La autoevaluación usa acc_test de la celda pre-escrita y N_ESTIMATORS/MAX_DEPTH/LEARNING_RATE de la sección 5
- Usa labels=[0, 1, 2] en confusion_matrix
- La autoevaluación reutiliza acc_test de la celda pre-escrita

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo y_test, y_pred (predicciones del XGBoost en test).
Genera código que:
1) calcule cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
2) dibuje heatmap con sns.heatmap (annot=True, ejes Real/Predicho)
3) imprima classification_report(y_test, y_pred, digits=3)
Incluye plt.show().
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · cm

# Tu código debe:
#   1. Matriz de confusión 3 clases
#   2. Heatmap etiquetado
#   3. classification_report

# Nota: acc_test ya está definido en la celda pre-escrita anterior.



In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `N_ESTIMATORS`, `MAX_DEPTH`, `LEARNING_RATE`
r = []
try:
    r = verificar_xgboost(acc_test, N_ESTIMATORS, MAX_DEPTH, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Métricas', r)


## Pregunta 7 — xAI global (1): importancia del booster y permutation

### 📘 Subpreguntas
- **7.a** ¿Coincide el top sensor con la física del daño (Strain)?
- **7.b** ¿Importancia del booster y permutation importance miden lo mismo?
- **7.c** ¿En qué se diferencian de SHAP (sección 8)?


In [ ]:
# --- PRE-ESCRITO: dos técnicas xAI globales ---
imp_series = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
imp_series.plot(kind='barh', ax=ax, color='#8e44ad')
ax.set_xlabel('Importancia (gain/weight)')
ax.set_title('xAI global — Importancia del booster (XGBoost)')
plt.tight_layout()
plt.show()

perm = permutation_importance(modelo, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE)
perm_series = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
perm_series.plot(kind='barh', ax=ax, color='#16a085')
ax.set_xlabel('Caída media en score al permutar')
ax.set_title('xAI global — Permutation importance')
plt.tight_layout()
plt.show()


### 🤖 Guía IA — Sección 7: xAI global — importancia del booster

**Objetivo:** Identificar el top-3 de features por importancia del booster (la celda pre-escrita ya muestra permutation).

**Tu código debe lograr**
- Calcular importancias desde modelo.feature_importances_
- Definir TOP3_IMPORTANCIA (lista de 3 nombres)
- Imprimir el top-3

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `TOP3_IMPORTANCIA`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- imp_series ya está graficado en la celda pre-escrita
- Puedes reutilizar imp_series o recalcular desde modelo
- Se espera que Strain (με) esté en el top-3 (coherencia física)

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo modelo (XGBoost entrenado) y FEATURES (lista de 5 sensores).
Genera código que:
1) ordene importancias en una Series indexada por FEATURES
2) defina TOP3_IMPORTANCIA = lista con los 3 nombres más importantes
3) imprima TOP3_IMPORTANCIA
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · TOP3_IMPORTANCIA

# Tu código debe:
#   1. Ordenar feature_importances_
#   2. Extraer top-3 nombres
#   3. Imprimir resultado



In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `TOP3_IMPORTANCIA`
r = []
try:
    r = verificar_importancia_global(TOP3_IMPORTANCIA)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Importancia global', r)


## Pregunta 8 — xAI con SHAP global (summary plot)

### 📘 Subpreguntas
- **8.a** ¿Qué clase de daño explicas con `CLASE_SHAP`?
- **8.b** ¿Qué feature aparece con mayor |SHAP| en promedio?


In [ ]:
# --- PRE-ESCRITO: TreeExplainer ---
def shap_values_clase(shap_values, clase: int):
    if isinstance(shap_values, list):
        return shap_values[clase]
    return shap_values[:, :, clase]

explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    n_clases_shap, n_casos_shap = len(shap_values), shap_values[0].shape[0]
else:
    n_casos_shap, _, n_clases_shap = shap_values.shape
print(f"SHAP: {n_clases_shap} clases × {n_casos_shap} casos test")


### 🤖 Guía IA — Sección 8: SHAP global

**Objetivo:** Summary plot de SHAP para una clase de daño.

**Tu código debe lograr**
- Definir CLASE_SHAP (0, 1 o 2)
- Graficar shap.summary_plot para esa clase
- Título que indique la clase explicada

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `CLASE_SHAP`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- explainer, shap_values y la función shap_values_clase() ya están en la celda pre-escrita
- Usa shap_values_clase(shap_values, CLASE_SHAP) — no indexes shap_values[CLASE_SHAP] directo
- CLASE_SHAP=2 explica daño severo (recomendado para alertas)

**Prompt listo para copiar** (sección avanzada — úsalo tal cual y adapta solo si la IA falla):

```text
En Jupyter tengo modelo, X_test, FEATURES, shap_values y la función:

def shap_values_clase(shap_values, clase):
    if isinstance(shap_values, list):
        return shap_values[clase]
    return shap_values[:, :, clase]

Genera código que:
1) defina CLASE_SHAP = 2
2) cree figura plt.figure(figsize=(8, 5))
3) llame shap.summary_plot(shap_values_clase(shap_values, CLASE_SHAP), X_test, feature_names=FEATURES, show=False)
4) ponga título con la clase y plt.show()
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · CLASE_SHAP

# Tu código debe:
#   1. Elegir CLASE_SHAP (0, 1 o 2)
#   2. Summary plot con shap_values_clase()
#   3. Título y plt.show()

# Nota: Usa shap_values_clase() de la celda pre-escrita.



In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `CLASE_SHAP`
r = []
try:
    r = verificar_shap_global(CLASE_SHAP)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — SHAP global', r)


## Pregunta 9 — xAI local con SHAP (waterfall)

### 📘 Subpreguntas
- **9.a** ¿La predicción del caso coincide con la etiqueta real?
- **9.b** ¿Qué sensor empujó más la predicción en ese instante?
- **9.c** ¿Cómo se comparará con LIME en la sección 10?


In [ ]:
# --- PRE-ESCRITO: utilidad para índice en test ---
def etiqueta_en_test(idx: int) -> tuple[int, int]:
    yt = int(y_test[idx])
    yp = int(y_pred[idx])
    return yt, yp


### 🤖 Guía IA — Sección 9: SHAP local (waterfall)

**Objetivo:** Explicar una predicción individual del conjunto de test.

**Tu código debe lograr**
- Definir INDEX_CASO válido en test
- Obtener y_true_caso y y_pred_caso
- Graficar waterfall SHAP para ese caso

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `INDEX_CASO`, `y_true_caso`, `y_pred_caso`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- etiqueta_en_test(idx) ya está definida en la celda pre-escrita
- Usa CLASE_SHAP de la sección anterior para elegir qué clase explicar
- Para expected_value multiclass, si es lista/array toma el índice CLASE_SHAP
- INDEX_CASO debe estar entre 0 y len(X_test)-1

**Prompt listo para copiar** (sección avanzada — úsalo tal cual y adapta solo si la IA falla):

```text
En Jupyter tengo X_test, y_test, y_pred, shap_values, explainer, FEATURES, CLASE_SHAP
y la función etiqueta_en_test(idx) que devuelve (y_true, y_pred).

También existe shap_values_clase(shap_values, clase).

Genera código que:
1) defina INDEX_CASO = 0
2) obtenga y_true_caso, y_pred_caso = etiqueta_en_test(INDEX_CASO) e imprima ambos
3) tome sv = shap_values_clase(shap_values, CLASE_SHAP)[INDEX_CASO]
4) base = explainer.expected_value; si es lista o ndarray, base = base[CLASE_SHAP]
5) cree exp = shap.Explanation(values=sv, base_values=base, data=X_test[INDEX_CASO], feature_names=FEATURES)
6) llame shap.plots.waterfall(exp, max_display=6, show=False) y plt.show()
```

**Prompt alternativo válido** (misma sección, otros parámetros permitidos):

```text
Prueba INDEX_CASO=5 si quieres otro caso; imprime si acertó o no antes del gráfico.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · INDEX_CASO
#   · y_true_caso
#   · y_pred_caso

# Tu código debe:
#   1. Elegir INDEX_CASO
#   2. Imprimir etiquetas real y predicha
#   3. Waterfall SHAP del caso

# Nota: Reutiliza CLASE_SHAP, shap_values_clase y etiqueta_en_test.



In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `INDEX_CASO`, `y_true_caso`, `y_pred_caso`
r = []
try:
    r = verificar_shap_local(INDEX_CASO, len(X_test), y_true_caso, y_pred_caso)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — SHAP local', r)


## Pregunta 10 — xAI local con LIME (mismo caso que SHAP)

**LIME** (Local Interpretable Model-agnostic Explanations) aproxima el modelo **solo alrededor de un caso**.
Usa el mismo `INDEX_CASO` y `CLASE_SHAP` que en la sección 9 para **comparar** con el waterfall de SHAP.

### 📘 Subpreguntas
- **10.a** ¿LIME y SHAP coinciden en el sensor más influyente?
- **10.b** ¿Por qué LIME puede variar si repites la explicación?


In [ ]:
# --- PRE-ESCRITO: explainer LIME tabular ---
explainer_lime = LimeTabularExplainer(
    X_train,
    feature_names=FEATURES,
    class_names=['Normal (0)', 'Daño menor (1)', 'Daño severo (2)'],
    mode='classification',
    random_state=RANDOM_STATE,
)
print("✅ LIME TabularExplainer listo (fondo = train).")


### 🤖 Guía IA — Sección 10: LIME local

**Objetivo:** Explicar el mismo caso de test con LIME y comparar con SHAP (sección 9).

**Tu código debe lograr**
- Usar el mismo INDEX_CASO y CLASE_SHAP
- Llamar explainer_lime.explain_instance con modelo.predict_proba
- Graficar con as_pyplot_figure(label=CLASE_SHAP)
- Definir TOP_LIME_FEATURES (3 nombres de sensor)

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `TOP_LIME_FEATURES`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- explainer_lime ya está creado en la celda pre-escrita
- Reutiliza INDEX_CASO de la sección 9 (ejecútala antes)
- LIME devuelve umbrales (ej. 'Strain (με) <= -0.68') — normaliza a nombre de sensor

**Prompt listo para copiar** (sección avanzada — úsalo tal cual y adapta solo si la IA falla):

```text
En Jupyter tengo explainer_lime, modelo, X_test, INDEX_CASO, CLASE_SHAP, FEATURES.
Genera código que:
1) exp_lime = explainer_lime.explain_instance(X_test[INDEX_CASO], modelo.predict_proba, num_features=5, labels=(CLASE_SHAP,))
2) fig = exp_lime.as_pyplot_figure(label=CLASE_SHAP); plt.title(...); plt.show()
3) TOP_LIME_FEATURES = [next((f for f in FEATURES if feat.startswith(f)), feat.split('<=')[0].strip()) for feat, _ in exp_lime.as_list(label=CLASE_SHAP)[:3]]
4) print(TOP_LIME_FEATURES)
Nota: LIME devuelve umbrales (ej. "Strain (με) <= -0.68"); normaliza al nombre del sensor.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · TOP_LIME_FEATURES

# Tu código debe:
#   1. explain_instance en INDEX_CASO
#   2. Gráfico LIME para CLASE_SHAP
#   3. Extraer TOP_LIME_FEATURES (3 sensores)

# Nota: Mismo INDEX_CASO que SHAP local (sección 9).



In [ ]:
# --- Autoevaluación 10 ---
# Requiere (celda «Aquí coloca tu solución»): `TOP_LIME_FEATURES`
r = []
try:
    r = verificar_lime_local(INDEX_CASO, len(X_test), TOP_LIME_FEATURES)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('10 — LIME local', r)


## Pregunta 11 — PDP y SHAP dependence (efecto marginal)

### 📘 Subpreguntas
- **10.a** ¿Qué muestra el PDP frente al dependence plot de SHAP?
- **10.b** ¿La relación de `FEATURE_PDP` con el daño es monótona?


In [ ]:
# --- PRE-ESCRITO: PDP para Strain ---
idx_strain = FEATURES.index('Strain (με)')
fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(
    modelo, X_test, [idx_strain], feature_names=FEATURES, target=2, ax=ax,
)
ax.set_title('PDP — Strain (με)')
plt.tight_layout()
plt.show()


### 🤖 Guía IA — Sección 11: PDP y SHAP dependence

**Objetivo:** Partial dependence y dependence plot para una feature adicional.

**Tu código debe lograr**
- Definir FEATURE_PDP (nombre de sensor válido)
- PDP con PartialDependenceDisplay y target=CLASE_SHAP
- SHAP dependence_plot para la misma feature

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `FEATURE_PDP`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- modelo es multiclass: PDP requiere target=CLASE_SHAP
- Usa shap_values_clase(shap_values, CLASE_SHAP) en dependence_plot
- La celda pre-escrita ya muestra PDP de Strain; tu tarea es otra feature (ej. Temp)

**Prompt listo para copiar** (sección avanzada — úsalo tal cual y adapta solo si la IA falla):

```text
En Jupyter tengo modelo, X_test, FEATURES, shap_values, CLASE_SHAP
y shap_values_clase(shap_values, clase).

Genera código que:
1) defina FEATURE_PDP = "Temp (°C)"
2) idx_pdp = FEATURES.index(FEATURE_PDP)
3) PDP: PartialDependenceDisplay.from_estimator(modelo, X_test, [idx_pdp], feature_names=FEATURES, target=CLASE_SHAP, ax=ax) con título
4) dependence: shap.dependence_plot(idx_pdp, shap_values_clase(shap_values, CLASE_SHAP), X_test, feature_names=FEATURES, show=False)
Incluye plt.show() en cada gráfico.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · FEATURE_PDP
#   · idx_pdp

# Tu código debe:
#   1. Elegir FEATURE_PDP
#   2. Gráfico PDP con target=CLASE_SHAP
#   3. Gráfico SHAP dependence

# Nota: Multiclass: no olvides target=CLASE_SHAP en el PDP.



In [ ]:
# --- Autoevaluación 11 ---
# Requiere (celda «Aquí coloca tu solución»): `FEATURE_PDP`
r = []
try:
    r = verificar_pdp(FEATURE_PDP)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('11 — PDP y dependence', r)


## Cierre y entrega (vía IA)

### ✍️ Reflexión final (3 frases)
- La variable que más impactó fue ___ porque ___.
- Para reducir costos en planta usaría el modelo para ___.
- Antes de obra real validaría con ___.

### 📋 Bitácora de prompts (obligatorio)

Completa [`prompts_entregados.md`](prompts_entregados.md): por cada sección, copia el prompt enviado, un resumen de la respuesta de la IA y qué aceptaste o rechazaste.

---
**Checklist:** 11 autoevaluaciones en ✅ · comparaste SHAP vs LIME · bitácora en prompts_entregados.md · bitácora entregada · gráficos revisados.
